# 📄 Approach 5 — Local LLM PDF QA (Ollama + Mistral)

Fully offline, fully free. No API keys, no internet calls after setup.

## How it works
```
PDF → Chunks → Retrieve relevant chunks (TF-IDF/embeddings)
    → Feed chunks + question into LOCAL Mistral LLM
    → LLM generates a natural language answer
    → If nothing relevant found → says "not in my knowledge"
```

Unlike Approach 1/3 which just return a raw chunk of text, this approach
actually **generates a proper answer** in natural language — like ChatGPT,
but running 100% on your machine.

## ⚠️ Important — Enable GPU first!
Go to: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**
(Free tier includes limited GPU hours — without it, this will be slow but still works)

## Steps
| Cell | Task |
|------|------|
| 1 | Install Ollama |
| 2 | Start Ollama server in background |
| 3 | Download Mistral model (~4GB, one-time) |
| 4 | Test Ollama is working |
| 5 | Install Python libraries & upload PDFs |
| 6 | Extract & chunk PDFs |
| 7 | Build retrieval index (finds relevant chunks) |
| 8 | RAG pipeline — retrieve + generate with local LLM |
| 9 | Interactive chatbot |

In [ ]:
# ─── Cell 1: Install Ollama ───────────────────────────────────────────────────
# Ollama is a tool that runs LLMs locally — like Docker, but for AI models.
# This installs the Ollama binary onto the Colab VM.
#
# Colab's base image is missing 'zstd' (a compression tool Ollama's installer
# needs to extract its files), so we install that first.

#!apt-get install -y zstd
#!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ─── Cell 2: Start Ollama Server in Background ───────────────────────────────
#
# Ollama normally runs as a background service (daemon). Colab doesn't give us
# normal OS service management, so we start it manually with nohup and "&"
# which keeps it running in the background while we use other cells.

#import subprocess
#import time

# Start the Ollama server as a background process
#process = subprocess.Popen(
#    ['ollama', 'serve'],
#    stdout=subprocess.DEVNULL,
#    stderr=subprocess.DEVNULL
#)

#print('⏳ Starting Ollama server...')
#time.sleep(5)  # give it a few seconds to boot up
#print('✅ Ollama server should be running now (PID:', process.pid, ')')

In [ ]:
# ─── Cell 3: Download Mistral Model ───────────────────────────────────────────
#
# This downloads Mistral 7B (quantized, ~4GB). One-time download.
# This is a STRONG open-source model that runs well even without huge GPUs.
#
# Other options you could swap in:
#   'llama3'      → Meta's Llama 3 (8B), also great
#   'phi3'        → Microsoft's small but capable model (~2GB, fastest)
#   'gemma2'      → Google's open model

#!ollama pull mistral

In [8]:
# ─── Cell 4: Test Ollama Is Working ───────────────────────────────────────────
import requests

response = requests.post(
    'http://localhost:11434/api/generate',
    json={
        'model': 'mistral',
        'prompt': 'Say hello in one short sentence.',
        'stream': False
    }
)

result = response.json()
print('✅ Ollama is working!\n')
print('Test response:', result['response'])

✅ Ollama is working!

Test response:  Hello there! How can I help you today?


In [9]:
# ─── Cell 5: Load PDFs via file picker ───────────────────────────────────────
import tkinter as tk
from tkinter import filedialog
import os

# Open a file picker dialog — user can select multiple PDFs
root = tk.Tk()
root.withdraw()  # hide the empty tkinter window
root.attributes('-topmost', True)  # bring dialog to front

print("📂 Opening file picker... (check your taskbar if it doesn't appear)")

selected_files = filedialog.askopenfilenames(
    title="Select PDF files",
    filetypes=[("PDF files", "*.pdf"), ("All files", "*.*")]
)

pdf_paths = list(selected_files)

if pdf_paths:
    print(f"\n✅ {len(pdf_paths)} PDF(s) selected:")
    for p in pdf_paths:
        print(f"   • {os.path.basename(p)}")
else:
    print("\n⚠️ No files selected. Run this cell again to pick files.")

📂 Opening file picker... (check your taskbar if it doesn't appear)

✅ 1 PDF(s) selected:
   • e-GP System User Manual - Procuring Entity User.pdf


In [12]:
# ─── Cell 6: Extract & Chunk PDFs ────────────────────────────────────────────
import fitz
import re

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text()
        text = re.sub(r'\s+', ' ', text).strip()
        if len(text) > 50:
            pages.append(text)
    return pages

def chunk_text(pages, source, chunk_size=200, overlap=40):
    # Slightly bigger chunks than before — gives the LLM more context per chunk
    chunks = []
    for page_num, page_text in enumerate(pages):
        words = page_text.split()
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if len(chunk.strip()) > 40:
                chunks.append({
                    'id': len(chunks),
                    'text': chunk,
                    'source': source,
                    'page': page_num + 1
                })
    return chunks

all_chunks = []
print('📖 Extracting and chunking PDFs...')
for path in pdf_paths:
    pages = extract_text(path)
    chunks = chunk_text(pages, source=path)
    all_chunks.extend(chunks)
    print(f'   {path}: {len(pages)} pages → {len(chunks)} chunks')

print(f'\n✅ Total chunks: {len(all_chunks)}')

📖 Extracting and chunking PDFs...
   C:/Users/ruhit/Downloads/e-GP System User Manual - Procuring Entity User.pdf: 542 pages → 613 chunks

✅ Total chunks: 613


In [14]:
# ─── Cell 7: Build Retrieval Index (Sentence Transformers) ──────────────────
#
# Even with a local LLM, we still need RETRIEVAL first. The LLM can't read
# your entire PDF every time (too slow, too much text). So we:
#   1. Find the most relevant chunks using semantic search (same as Approach 1)
#   2. Only send THOSE chunks to the LLM, not the whole document
# This is the "R" in RAG (Retrieval-Augmented Generation).

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print('🔧 Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('⚙️  Encoding all chunks...')
chunk_texts = [c['text'] for c in all_chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

def retrieve_chunks(query, top_k=4):
    query_emb = embedder.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_emb, chunk_embeddings).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    results = [(all_chunks[i], scores[i]) for i in top_indices]
    return results

print('\n✅ Retrieval index ready.')

🔧 Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⚙️  Encoding all chunks...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]


✅ Retrieval index ready.


In [15]:
# ─── Cell 8: RAG Pipeline — Retrieve + Generate with Local LLM ───────────────
#
# This is the full RAG loop:
#   1. RETRIEVE: find chunks most similar to the question
#   2. CHECK: if similarity too low, don't even ask the LLM — say "not in knowledge"
#   3. AUGMENT: build a prompt with those chunks as context
#   4. GENERATE: send prompt to local Mistral via Ollama, get a real answer

import requests

SIMILARITY_THRESHOLD = 0.25   # below this, we don't trust any chunk
TOP_K = 4                      # number of chunks to feed the LLM

def ask_local_llm(question, context):
    """Send a RAG prompt to Mistral running locally via Ollama."""
    prompt = f"""You are a helpful assistant that answers ONLY using the provided context below.
If the answer is not in the context, say exactly: \"This information is not in my knowledge base.\"
Do not use any outside knowledge. Keep your answer concise.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = requests.post(
        'http://localhost:11434/api/generate',
        json={
            'model': 'mistral',
            'prompt': prompt,
            'stream': False,
            'options': {
                'temperature': 0.2,   # low temperature = more focused, less creative
                'num_predict': 300    # max tokens in the answer
            }
        }
    )
    return response.json()['response'].strip()

def rag_answer(question):
    # Step 1: Retrieve
    results = retrieve_chunks(question, top_k=TOP_K)
    best_score = results[0][1] if results else 0

    # Step 2: Check relevance
    if best_score < SIMILARITY_THRESHOLD:
        return 'This information is not in my knowledge base.', [], best_score

    # Step 3: Augment — build context string from retrieved chunks
    context = '\n\n---\n\n'.join([f"[{c['source']}]\n{c['text']}" for c, score in results])
    sources = list(set(c['source'] for c, score in results))

    # Step 4: Generate — ask the local LLM
    answer = ask_local_llm(question, context)
    return answer, sources, best_score

# Quick test
print('🧪 Testing the RAG pipeline...\n')
test_q = 'What is this document about?'
answer, sources, score = rag_answer(test_q)
print(f'Q: {test_q}')
print(f'A: {answer}')
print(f'Sources: {sources}')
print(f'Confidence: {score:.3f}')

🧪 Testing the RAG pipeline...

Q: What is this document about?
A: The document is about the e-GP System User Manual for the Procuring Entity User, which provides instructions on how to prepare and manage procurement documents within the e-GP system. It includes details about Guidance Notes, Sections, Forms, Drawings, notices, and other relevant aspects of tender document preparation.
Sources: ['C:/Users/ruhit/Downloads/e-GP System User Manual - Procuring Entity User.pdf']
Confidence: 0.411


In [ ]:
# ─── Cell 9: Interactive Chatbot ──────────────────────────────────────────────

print('💬 Local LLM PDF Chatbot ready! (Mistral via Ollama — 100% offline)')
print('   Type your question. Type "exit" to quit.\n')
print('-' * 60)

while True:
    question = input('\nYou: ').strip()
    if question.lower() in ['exit', 'quit', 'q']:
        print('👋 Goodbye!')
        break
    if not question:
        continue

    print('🤖 Thinking...')
    answer, sources, score = rag_answer(question)

    print(f'\n🤖 Answer:')
    print(answer)
    if sources:
        print(f'\n📄 Source(s): {", ".join(sources)}')
    print(f'   (Relevance score: {score:.3f})')

💬 Local LLM PDF Chatbot ready! (Mistral via Ollama — 100% offline)
   Type your question. Type "exit" to quit.

------------------------------------------------------------



You:  do you know tender preparation process?


🤖 Thinking...

🤖 Answer:
Yes, according to the provided context, the tender preparation process involves the following steps:

1. In the Tender menu, the PE clicks on “Create Tender” sub-menu if the APP Package is published on e-GP Portal.
2. Select Financial Year and APP from the drop-down boxes and click on “Search APP” button.
3. Published packages of that APP will be displayed in a grid. Select the package for which you want to create a tender and then click “Submit” button.
4. The "Steps for Tender Preparation" button takes you to a PDF file where steps for Tender preparation are available.

📄 Source(s): C:/Users/ruhit/Downloads/e-GP System User Manual - Procuring Entity User.pdf
   (Relevance score: 0.565)



You:  give me the document preparation process details


🤖 Thinking...

🤖 Answer:
The document preparation process, as per the provided context, involves the following steps for both works and services procurement:

1. Document Preparation using STD e-GP: This includes filling Tender Data Sheet (TDS), Particular Conditions of Contract (PCC), and other forms specific to each Standard (STD).

2. For works, fill the Fixed Rate BOQ Form and BOQ for Salvage Work. For services, fill Proposal/Tender Data Sheet (PDS/TDS) and Particular Conditions of Contract (PCC).

3. Fill other necessary forms and schedules as part of the bid, which may include forms for technical specification, pricing, project implementation schedules, declarations, and any other required documentation.

4. In some cases, there might be a drawing section to include visual representations such as technical diagrams or architectural plans.

5. The document preparation is done by the Procuring Entity/Authorized User (PE/AU) and can be edited to reflect the project details, technica